# Task 43: Manual Model Stacking

Tune scratch base learners on a separate training partition, create out-of-fold base predictions for meta-training, then evaluate the stacked classifier on an untouched test split.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [Path('heart.csv'), Path('Task_43_Manual_Model_Stacking/heart.csv'), Path('Heart_Disease_Tasks/Task_43_Manual_Model_Stacking/heart.csv')]
data = pd.read_csv(next(p for p in candidates if p.is_file())).drop_duplicates().reset_index(drop=True)
features = [c for c in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'] if c in data]
X, y = data[features].to_numpy(float), data.target.to_numpy(int)


def split_indices(labels, fraction, seed):
    rng = np.random.default_rng(seed)
    first, second = [], []
    for label in np.unique(labels):
        indices = np.flatnonzero(labels == label)
        rng.shuffle(indices)
        cut = round(fraction * len(indices))
        first.extend(indices[:cut]); second.extend(indices[cut:])
    rng.shuffle(first); rng.shuffle(second)
    return np.array(first), np.array(second)


train_indices, test_indices = split_indices(y, .8, 43)
tune_indices, stack_indices = split_indices(y[train_indices], .2, 44)
tune_indices, stack_indices = train_indices[tune_indices], train_indices[stack_indices]
X_tune, y_tune = X[tune_indices], y[tune_indices]
X_stack, y_stack = X[stack_indices], y[stack_indices]
X_test, y_test = X[test_indices], y[test_indices]


class Logistic:
    def __init__(self, regularization): self.regularization = regularization
    def fit(self, x, labels, steps=900, rate=.06):
        self.weights = np.zeros(x.shape[1]); self.bias = 0.0
        for _ in range(steps):
            probability = 1 / (1 + np.exp(-np.clip(x @ self.weights + self.bias, -35, 35)))
            error = probability - labels
            self.weights -= rate * (x.T @ error / len(labels) + self.regularization * self.weights)
            self.bias -= rate * error.mean()
        return self
    def predict_proba(self, x): return 1 / (1 + np.exp(-np.clip(x @ self.weights + self.bias, -35, 35)))


class DecisionTree:
    def __init__(self, depth): self.depth = depth
    def fit(self, x, labels): self.root = self._grow(x, labels, 0); return self
    def _grow(self, x, labels, depth):
        counts = np.bincount(labels, minlength=2)
        node = {'prediction': int(np.argmax(counts)), 'probability': float(labels.mean())}
        if depth >= self.depth or len(labels) < 8 or len(np.unique(labels)) < 2: return node
        parent = 1 - np.sum((counts / len(labels)) ** 2)
        best = None
        for column in range(x.shape[1]):
            values = np.unique(x[:, column])
            for threshold in (values[:-1] + values[1:]) / 2:
                left = x[:, column] <= threshold
                if not left.any() or left.all(): continue
                left_counts = np.bincount(labels[left], minlength=2) / left.sum()
                right_counts = np.bincount(labels[~left], minlength=2) / (~left).sum()
                gain = parent - left.mean() * (1 - np.sum(left_counts ** 2)) - (~left).mean() * (1 - np.sum(right_counts ** 2))
                if best is None or gain > best[0]: best = (gain, column, threshold, left)
        if best is None or best[0] <= 0: return node
        _, column, threshold, left = best
        node.update(column=column, threshold=threshold, left=self._grow(x[left], labels[left], depth + 1), right=self._grow(x[~left], labels[~left], depth + 1))
        return node
    def predict_proba(self, x):
        predictions = []
        for row in x:
            node = self.root
            while 'column' in node: node = node['left'] if row[node['column']] <= node['threshold'] else node['right']
            predictions.append(node['probability'])
        return np.array(predictions)


def make_base(name, setting, x, labels):
    if name == 'logistic': return Logistic(setting).fit(x, labels)
    return DecisionTree(setting).fit(x, labels)


def f1_score(actual, predicted):
    tp = np.sum((actual == 1) & (predicted == 1)); fp = np.sum((actual == 0) & (predicted == 1)); fn = np.sum((actual == 1) & (predicted == 0))
    precision = tp / (tp + fp) if tp + fp else 0.0; recall = tp / (tp + fn) if tp + fn else 0.0
    return float(2 * precision * recall / (precision + recall)) if precision + recall else 0.0


# Optimize base-model settings on a tuning subset kept out of meta-training.
tune_train_indices, tune_valid_indices = split_indices(y_tune, .8, 45)
selected, tuning_results = {}, []
for name, choices in [('logistic', [.001, .03]), ('tree', [3, 5])]:
    best_score = -1.0
    for setting in choices:
        model = make_base(name, setting, X_tune[tune_train_indices], y_tune[tune_train_indices])
        score = f1_score(y_tune[tune_valid_indices], (model.predict_proba(X_tune[tune_valid_indices]) >= .5).astype(int))
        tuning_results.append((name, setting, score))
        if score > best_score: best_score, selected[name] = score, setting
print('Tuning results:', tuning_results)
print('Selected base settings:', selected)

# Build all four folds up front; each training row receives a prediction from a model that did not fit on it.
rng = np.random.default_rng(46)
fold_lists = [[] for _ in range(4)]
for label in np.unique(y_stack):
    label_indices = np.flatnonzero(y_stack == label)
    rng.shuffle(label_indices)
    for fold_index, part in enumerate(np.array_split(label_indices, 4)):
        fold_lists[fold_index].extend(part.tolist())
folds = [np.array(sorted(part), dtype=int) for part in fold_lists]
meta_features = np.zeros((len(y_stack), 2))
for validation_indices in folds:
    validation_set = set(validation_indices.tolist())
    training_indices = np.array([i for i in range(len(y_stack)) if i not in validation_set])
    fold_train, fold_valid = X_stack[training_indices], X_stack[validation_indices]
    means, scales = fold_train.mean(axis=0), fold_train.std(axis=0); scales[scales == 0] = 1.0
    fold_train, fold_valid = (fold_train - means) / scales, (fold_valid - means) / scales
    for column, name in enumerate(['logistic', 'tree']):
        model = make_base(name, selected[name], fold_train, y_stack[training_indices])
        meta_features[validation_indices, column] = model.predict_proba(fold_valid)

# Fit the meta learner only on out-of-fold probabilities.
meta_means, meta_scales = meta_features.mean(axis=0), meta_features.std(axis=0)
meta_scales[meta_scales == 0] = 1.0
meta_train = (meta_features - meta_means) / meta_scales
meta_model = Logistic(.02).fit(meta_train, y_stack, steps=1200, rate=.08)

# Refit each base learner on the full stack-training subset, then score the untouched test set.
means, scales = X_stack.mean(axis=0), X_stack.std(axis=0); scales[scales == 0] = 1.0
scaled_stack, scaled_test = (X_stack - means) / scales, (X_test - means) / scales
test_features = np.column_stack([make_base(name, selected[name], scaled_stack, y_stack).predict_proba(scaled_test) for name in ['logistic', 'tree']])
test_features = (test_features - meta_means) / meta_scales
test_predictions = (meta_model.predict_proba(test_features) >= .5).astype(int)
tp = np.sum((y_test == 1) & (test_predictions == 1)); tn = np.sum((y_test == 0) & (test_predictions == 0))
fp = np.sum((y_test == 0) & (test_predictions == 1)); fn = np.sum((y_test == 1) & (test_predictions == 0))
precision = tp / (tp + fp) if tp + fp else 0.0; recall = tp / (tp + fn) if tp + fn else 0.0
print('Held-out stacking metrics:', {'accuracy': float((tp + tn) / len(y_test)), 'precision': float(precision), 'recall': float(recall), 'f1': f1_score(y_test, test_predictions)})
print(f'Disjoint tuning rows: {len(y_tune)}; OOF meta-training rows: {len(y_stack)}; folds: {len(folds)}; untouched test rows: {len(y_test)}')
print('The meta-model uses only out-of-fold logistic and tree probabilities, never in-sample base predictions.')

Tuning results: [('logistic', 0.001, 0.7142857142857143), ('logistic', 0.03, 0.6153846153846154), ('tree', 3, 0.33333333333333337), ('tree', 5, 0.33333333333333337)]
Selected: {'logistic': 0.001, 'tree': 3}


IndexError: list index out of range